In [1]:
import IPython.core.pylabtools
import IPython.core.pylabtools
import os
import sys
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import mlflow
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import time
import itertools
from joblib import Parallel, delayed

# Ask TensorFlow to list all available physical GPUs
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print(f"✅ M3 Pro GPU ACTIVATED! Found: {gpu_devices}")
    # Optional: Set memory growth to prevent TF from hoarding all unified memory
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print("❌ GPU not found. TensorFlow is falling back to the CPU.")

✅ M3 Pro GPU ACTIVATED! Found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data

In [2]:
# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Spark setup
from dotenv import load_dotenv
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())

from src.common.setup_spark import create_spark_session
from config.config_spark import Paths

# MLflow Setup
mlflow.set_tracking_uri("sqlite:///mlflow.db") # Local SQLite database for tracking
experiment_name = "SP500_Momentum_Backtest"
mlflow.set_experiment(experiment_name)
print(f"MLflow Experiment set to: {experiment_name}")

spark = create_spark_session()
print("Spark Session created.")

# Load Data
df_gold = spark.read.format("delta").load(Paths.SP500_MOMENTUM_CRASH_WEEKLY_GOLD)
df_gold.createOrReplaceTempView("gold_prices")

df = df_gold.toPandas()

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.weekday

#df = df[df['bull_market']==1]

print(f"Data loaded: {df.shape}")
print(f"Years: {df['year'].unique().min()}")

2026-03-10 15:49:03.609 | INFO     | src.common.setup_spark:create_spark_session:19 - 🛠️ Configurant Spark avec le connecteur GCS : https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.6/gcs-connector-hadoop3-2.2.6-shaded.jar


MLflow Experiment set to: SP500_Momentum_Backtest


26/03/10 15:49:04 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 192.168.1.147 instead (on interface en0)
26/03/10 15:49:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-27a187ee-f6fb-4f50-891b-d44fba39b4f8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 69ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	| 

:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/03/10 15:49:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 15:49:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2026-03-10 15:49:06.684 | SUCCESS  | src.common.setup_spark:create_spark_session:38 - ✅ Spark Session 'SparkApp' créée avec succès ! (Version: 3.5.3)


Spark Session created.


Data loaded: (663262, 33)
Years: 1990


In [3]:
class OHLCResampler:
    def __init__(self, data, date_col="Date", ticker_col="Ticker"):
        self.data = data.copy()
        self.date_col = date_col
        self.ticker_col = ticker_col

        # S'assurer que la colonne de date est bien en datetime
        self.data[date_col] = pd.to_datetime(self.data[date_col])
        self.data.set_index(date_col, inplace=True)

    def resample(self, period='W'):
        grouped = self.data.groupby(self.ticker_col)

        resampled = grouped.resample(period).agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).reset_index()

        return resampled

# Backtest

## Top n Portfolio

In [12]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import itertools
import time
from joblib import Parallel, delayed

# ==========================================
# 🧠 1. LE MOTEUR BLACK-SCHOLES (100x Plus Rapide)
# ==========================================
def vectorized_black_scholes_call(S, K, T, r, sigma):
    """Calcule le Call pour des colonnes entières (Série Pandas) instantanément."""
    # Sécurités pour éviter les divisions par zéro
    T = np.maximum(T, 1e-6)
    sigma = np.maximum(sigma, 1e-6)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    call_price = (S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))
    return np.where((T <= 1e-6) | (sigma <= 1e-6) | (S <= 0), np.maximum(S - K, 0.0), call_price)

# ==========================================
# 🚀 2. LA STRATÉGIE PRINCIPALE
# ==========================================
def generate_top_n_portfolio_integrated(
    df, 
    momentum_window=52, 
    top_n=5, 
    rebalance_freq='W', 
    ma_fast_window=18, 
    ma_slow_window=30, 
    bull_strategy='leaps_option',
    option_allocation=0.05,      
    days_to_expiry=180,          # Pour les options classiques
    holding_days=7,              # Durée de détention (valable pour toutes les options)
    iv_markup=1.20,              
    spread_penalty=0.025,
    leaps_days_to_expiry=500,    # 👈 NOUVEAU : Jours avant expiration des LEAPS
    leaps_strike_ratio=0.80
    ):
    # --- 1. FILTRE MACRO (S&P 500) ---
    market_data = df[['date', 'adjClose_GSPC']].drop_duplicates().sort_values('date').copy()
    market_data['ma_fast'] = market_data['adjClose_GSPC'].rolling(window=ma_fast_window).mean()
    market_data['ma_slow'] = market_data['adjClose_GSPC'].rolling(window=ma_slow_window).mean()
    
    cond_bull = (market_data['ma_fast'] > market_data['ma_slow']) & (market_data['adjClose_GSPC'] > market_data['ma_slow'])
    cond_bear = (market_data['ma_fast'] <= market_data['ma_slow']) & (market_data['adjClose_GSPC'] > market_data['ma_fast'])
    market_data['go_to_cash'] = np.where(cond_bull | cond_bear, 0, 1)

    # --- 2. PRÉPARATION DES ACTIONS ---
    stock_data = df[df['symbol'] != '^GSPC'][['symbol', 'date', 'adjClose']].copy()
    data = stock_data.merge(market_data[['date', 'go_to_cash']], on='date', how='left').sort_values(['symbol', 'date'])

    # --- 3. MOMENTUM & VOLATILITÉ ---
    data['Momentum'] = data.groupby('symbol')['adjClose'].pct_change(periods=momentum_window)
    data['Daily_Return'] = data.groupby('symbol')['adjClose'].pct_change()
    
    data['Volatility_30D'] = data.groupby('symbol')['Daily_Return'].rolling(window=30).std().reset_index(0, drop=True) * np.sqrt(252)
    data['Volatility_30D'] = data['Volatility_30D'].fillna(0.15).clip(lower=0.15)
    data['Next_adjClose'] = data.groupby('symbol')['adjClose'].shift(-1)

    # ==========================================
    # 🔥 4. CALCUL DU RENDEMENT (STOCK vs OPTION vs LEAPS)
    # ==========================================
    if bull_strategy == 'stock':
        data['NextReturn'] = (data['Next_adjClose'] / data['adjClose']) - 1
        base_weight = 1.0 / top_n
        
    elif bull_strategy == 'call_option':
        data['Strike_Price'] = data['adjClose']
        data['Synthetic_IV'] = data['Volatility_30D'] * iv_markup
        
        t_buy = days_to_expiry / 365.0
        t_sell = (days_to_expiry - holding_days) / 365.0
        r = 0.04 
        
        data['Theo_Buy'] = vectorized_black_scholes_call(data['adjClose'], data['Strike_Price'], t_buy, r, data['Synthetic_IV'])
        data['Theo_Sell'] = vectorized_black_scholes_call(data['Next_adjClose'], data['Strike_Price'], t_sell, r, data['Synthetic_IV'])
        
        data['Premium_Paid'] = data['Theo_Buy'] * (1 + spread_penalty)
        data['Premium_Received'] = data['Theo_Sell'] * (1 - spread_penalty)
        
        data['Option_Return'] = np.where(
            data['Premium_Paid'] > 0,
            (data['Premium_Received'] - data['Premium_Paid']) / data['Premium_Paid'],
            0.0
        )
        data['NextReturn'] = np.maximum(data['Option_Return'], -1.0)
        base_weight = option_allocation 

    elif bull_strategy == 'leaps_option':
        # 🚀 L'utilisation de tes nouveaux paramètres !
        
        # Le Strike est calculé dynamiquement grâce au ratio
        data['Strike_Price'] = data['adjClose'] * leaps_strike_ratio 
        
        # Taxe modérée des Market Makers 
        data['Synthetic_IV'] = data['Volatility_30D'] * 1.10 
        
        # Le temps avant expiration utilise ton nouveau paramètre
        t_buy = leaps_days_to_expiry / 365.0
        
        # Le temps à la revente déduit les jours de détention (holding_days)
        t_sell = (leaps_days_to_expiry - holding_days) / 365.0
        r = 0.04 # Taux sans risque
        
        # Prix de la prime Aujourd'hui
        data['Theo_Buy'] = vectorized_black_scholes_call(
            data['adjClose'], data['Strike_Price'], t_buy, r, data['Synthetic_IV']
        )
        
        # Prix de la prime la Semaine Prochaine
        data['Theo_Sell'] = vectorized_black_scholes_call(
            data['Next_adjClose'], data['Strike_Price'], t_sell, r, data['Synthetic_IV']
        )
        
        # RENDEMENT HEBDOMADAIRE PUR 
        data['Option_Return'] = np.where(
            data['Theo_Buy'] > 0,
            (data['Theo_Sell'] - data['Theo_Buy']) / data['Theo_Buy'],
            0.0
        )
        
        # On bloque la perte maximale mathématique à -100%
        data['NextReturn'] = np.maximum(data['Option_Return'], -1.0)
        
        # Sizing sécurisé
        base_weight = option_allocation

    # --- 5. GESTION DES DATES DE REBALANCEMENT 🛠️ ---
# --- 5. GESTION DES DATES DE REBALANCEMENT 🛠️ ---
        if rebalance_freq == 'W':
            data['is_rebalance_date'] = True
        else:
            if rebalance_freq == 'M':
                data['Period'] = data['date'].dt.to_period('M')
            elif rebalance_freq == 'Q':
                data['Period'] = data['date'].dt.to_period('Q')
            elif rebalance_freq == '6M':
                # Création d'un flag "Semestre 1" ou "Semestre 2"
                data['Period'] = data['date'].dt.year.astype(str) + "H" + np.where(data['date'].dt.month <= 6, '1', '2')
            elif rebalance_freq == 'Y':
                data['Period'] = data['date'].dt.to_period('Y')
                
            rebalance_dates = data.groupby('Period')['date'].transform('max')
            data['is_rebalance_date'] = (data['date'] == rebalance_dates)

    # --- 6. RANKING & POIDS ---
    valid_data = data[data['is_rebalance_date']].dropna(subset=['Momentum']).copy()
    valid_data['Rank'] = valid_data.groupby('date')['Momentum'].rank(method='first', ascending=False)
    
    buys = valid_data[valid_data['Rank'] <= top_n].copy()
    buys['Base_Weight'] = base_weight

    # --- 7. FUSION FINALE ---
    data = data.merge(buys[['symbol', 'date', 'Base_Weight']], on=['symbol', 'date'], how='left')
    data.loc[data['is_rebalance_date'] & data['Base_Weight'].isna(), 'Base_Weight'] = 0.0
    data['Base_Weight'] = data.groupby('symbol')['Base_Weight'].ffill().fillna(0.0)
    
    data['Target_Weight'] = np.where(data['go_to_cash'] == 1, 0.0, data['Base_Weight'])
    data = data.dropna(subset=['NextReturn'])

    return data

## Vectorized Backtester

In [13]:
def run_vectorized_backtest(data, transaction_cost=0):
    """
    Calcule la performance globale du portefeuille avec les frais de transaction (Turnover).
    """
    # Calcul du Turnover : De combien le poids de l'action a-t-il changé ?
    data['Weight_Change'] = data.groupby('symbol')['Target_Weight'].diff().fillna(data['Target_Weight'])
    
    # On paie les frais sur la valeur absolue du changement (qu'on achète ou qu'on vende)
    data['Cost'] = data['Weight_Change'].abs() * transaction_cost
    
    # Rendement brut de chaque action
    data['Strat_Return'] = data['Target_Weight'] * data['NextReturn']
    
    # Agrégation au niveau du Portefeuille (Somme de toutes les actions pour une date donnée)
    port_returns = data.groupby('date')[['Strat_Return', 'Cost']].sum()
    
    # Rendement Net
    port_returns['Net_Return'] = port_returns['Strat_Return'] - port_returns['Cost']
    
    # Croissance du Capital
    port_returns['Capital'] = (1 + port_returns['Net_Return']).cumprod()
    
    return port_returns

## Running only one combination

In [14]:
def run_single_backtest(params, df_source, start_date, end_date):
    # 1. Nouveaux paramètres épurés (uniquement ce qui concerne le Stock Picking)
    mom_win, top_n, reb_freq, ma_fast_window, ma_slow_window, option_allocation,leaps_days_to_expiry, leaps_strike_ratio = params

    default_output = {
        "Momentum_Window": mom_win, 
        "Top_N": top_n,
        "Rebalance": reb_freq,
        "MA_fast_Window": ma_fast_window,
        "MA_slow_Window": ma_slow_window,
        "Total Return": np.nan, 
        "CAGR": np.nan,
        "Max Drawdown": np.nan,
        "Calmar Ratio": np.nan,
        "Sharpe Ratio": np.nan, 
        "Error": None
    }

    try:
        # 🛠️ CORRECTION : Le nom de la fonction est maintenant correct
        full_signals = generate_top_n_portfolio_integrated(
            df_source, 
            momentum_window=mom_win, 
            top_n=top_n, 
            rebalance_freq=reb_freq,
            ma_fast_window=ma_fast_window,
            ma_slow_window=ma_slow_window,
            option_allocation=option_allocation,
            leaps_days_to_expiry=leaps_days_to_expiry,
            leaps_strike_ratio=leaps_strike_ratio
        )

        # 3. Filtre des dates de test
        mask = (full_signals['date'] >= pd.to_datetime(start_date)) & (full_signals['date'] <= pd.to_datetime(end_date))
        backtest_data = full_signals.loc[mask]

        if backtest_data.empty:
            default_output["Error"] = "No data in timeframe"
            return default_output

        # 4. Lancement du Backtest Vectorisé 
        res_df = run_vectorized_backtest(backtest_data, transaction_cost=0.005)

        # 5. Calcul des métriques
        total_return = res_df['Capital'].iloc[-1] - 1
        n_years = (res_df.index[-1] - res_df.index[0]).days / 365.25
        cagr = (res_df['Capital'].iloc[-1]) ** (1 / max(1, n_years)) - 1
        
        rolling_max = res_df['Capital'].cummax()
        max_dd = ((res_df['Capital'] - rolling_max) / rolling_max).min()
        
        mean_ret = res_df['Net_Return'].mean()
        std_ret = res_df['Net_Return'].std()
        sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret > 0 else 0 
        calmar = abs(cagr/max_dd)

        # Mise à jour de l'output
        output = default_output.copy()
        output.update({
            "Total Return": total_return, 
            "CAGR": cagr, 
            "Max Drawdown": max_dd,
            "Calmar Ratio": calmar,
            "Sharpe Ratio": sharpe,
            "Option_Allocation": option_allocation,
            "Leaps_Days_to_Expiry": leaps_days_to_expiry,
            "Leaps_Strike_Ratio": leaps_strike_ratio
        })
        return output

    except Exception as e:
        default_output["Error"] = str(e)
        return default_output

## Grid Search

In [15]:
def grid_search_execution(df, param_grid, start_date, end_date):
    keys, values = zip(*param_grid.items())
    combinations = [v for v in itertools.product(*values)]

    print(f"🚀 Lancement de la Grid Search sur {len(combinations)} combinaisons...")
    start_time = time.time()

    # Exécution en parallèle sur tous les coeurs du processeur (n_jobs=-1)
    results_list = Parallel(n_jobs=-1)(
        delayed(run_single_backtest)(params, df, start_date, end_date) for params in combinations
    )

    end_time = time.time()
    print(f"✅ Terminé en {end_time - start_time:.2f} secondes.")

    results_df = pd.DataFrame(results_list)
    
    # On affiche les 10 meilleures stratégies triées par Sharpe Ratio
    best_strats = results_df[results_df['Error'].isna()].sort_values(by='Sharpe Ratio', ascending=False)
    return best_strats

## Running Backtest

In [ ]:
# Définition de la grille de paramètres (Stock Picking pur)
param_grid = {
    'momentum_window': [26, 52],   # 1 mois, 3 mois, 6 mois, 1 an
    'top_n': [5, 6],         # Portefeuille très concentré vs très diversifié
    'rebalance_freq': ['6M'],
    'ma_fast_window': [26], 
    'ma_slow_window': [55],
    'option_allocation': [0.05, 0.1, 0.15, 0.2],
    'leaps_days_to_expiry': [365, 500, 730],   # Teste des LEAPS à 1 an, 1.5 an et 2 ans
    'leaps_strike_ratio': [0.5, 0.6, 0.7, 0.8, 0.9]  # Fréquence de rotation du portefeuille
}

# Lancement de la Grid Search
best_strategies_df = grid_search_execution(
    df=df,  # Ton DataFrame source
    param_grid=param_grid,
    start_date='1980-01-01',
    end_date='2026-01-01'
)

🚀 Lancement de la Grid Search sur 720 combinaisons...


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_14064/1409827378.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_14064/1409827378.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_14064/1409827378.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

✅ Terminé en 96.85 secondes.


In [17]:
best_strategies_df.sort_values('CAGR', ascending=False).head(20)

,Momentum_Window,Top_N,Rebalance,MA_fast_Window,MA_slow_Window,Total Return,CAGR,Max Drawdown,Calmar Ratio,Sharpe Ratio,Error,Option_Allocation,Leaps_Days_to_Expiry,Leaps_Strike_Ratio
535,52,5,Y,26,55,113.019737,0.140727,-0.601730,0.233871,0.531189,None,0.20,730,0.5
715,52,6,Y,26,55,91.290004,0.134042,-0.676978,0.198001,0.484758,None,0.20,730,0.5
536,52,5,Y,26,55,86.859433,0.132492,-0.643503,0.205892,0.500598,None,0.20,730,0.6
530,52,5,Y,26,55,80.528167,0.130140,-0.672851,0.193416,0.488725,None,0.20,500,0.5
537,52,5,Y,26,55,65.924361,0.123956,-0.679813,0.182338,0.470917,None,0.20,730,0.7
716,52,6,Y,26,55,62.235398,0.122186,-0.719532,0.169813,0.451431,None,0.20,730,0.6
525,52,5,Y,26,55,58.236064,0.120149,-0.727400,0.165177,0.454267,None,0.20,365,0.5
710,52,6,Y,26,55,56.827676,0.119400,-0.743796,0.160528,0.441730,None,0.20,500,0.5
655,52,6,6M,26,55,53.632769,0.117633,-0.649135,0.181216,0.444026,None,0.20,730,0.5
475,52,5,6M,26,55,50.933609,0.116060,-0.555640,0.208877,0.462658,None,0.20,730,0.5


In [18]:
best_strategies_df.sort_values('Calmar Ratio', ascending=False).head(20)

,Momentum_Window,Top_N,Rebalance,MA_fast_Window,MA_slow_Window,Total Return,CAGR,Max Drawdown,Calmar Ratio,Sharpe Ratio,Error,Option_Allocation,Leaps_Days_to_Expiry,Leaps_Strike_Ratio
520,52,5,Y,26,55,45.709502,0.112776,-0.479695,0.235099,0.531189,None,0.15,730,0.5
535,52,5,Y,26,55,113.019737,0.140727,-0.601730,0.233871,0.531189,None,0.20,730,0.5
505,52,5,Y,26,55,14.736918,0.079625,-0.343850,0.231570,0.531189,None,0.10,730,0.5
490,52,5,Y,26,55,3.368584,0.041839,-0.184250,0.227079,0.531189,None,0.05,730,0.5
430,52,5,6M,26,55,2.626467,0.036461,-0.163322,0.223247,0.462658,None,0.05,730,0.5
445,52,5,6M,26,55,9.776439,0.068321,-0.307087,0.222481,0.462658,None,0.10,730,0.5
460,52,5,6M,26,55,25.194377,0.095027,-0.432136,0.219900,0.462658,None,0.15,730,0.5
506,52,5,Y,26,55,13.178495,0.076500,-0.355837,0.214986,0.500598,None,0.10,730,0.6
491,52,5,Y,26,55,3.174076,0.040521,-0.191391,0.211719,0.500598,None,0.05,730,0.6
521,52,5,Y,26,55,38.171578,0.107345,-0.510722,0.210182,0.500598,None,0.15,730,0.6
